<div>
<center><img src="../assets/Flux-logo.svg" width="360"/></center>
</div>

<div style="background:linear-gradient(90deg,#036291 0%,#91C2D8 100%);padding:20px 26px;border-radius:10px;border-left:10px solid #D9A441;margin-top:18px">
<h1 style="margin:0;color:#ffffff">Module 2: Hybrid Quantum-Classical Under Flux</h1>
<p style="margin:6px 0 0 0;color:#DCECF4;font-size:15px">A local simulator, no cloud account, no hardware queue</p>
<p style="margin:2px 0 0 0;color:#DCECF4;font-size:13px">SC26 &middot; Chicago &middot; November 2026</p>
</div>

So far Flux has scheduled classical HPC, AI/ML training, and a Kubernetes control plane.
Here we add a quantum workload, and the point is that **nothing about Flux changes**. A
hybrid quantum-classical algorithm is a classical optimizer wrapped around a circuit
evaluation. That loop is a job. Flux schedules jobs.

Everything runs on the CPU cores Flux gave you, via `qiskit-aer`. No account, no
credentials, no waiting in a hardware queue.


## Background: QAOA max-cut

**Max-cut** asks: given a graph, split its nodes into two groups so that as many edges as
possible run *between* the groups rather than inside them. It is NP-hard.

**QAOA** (the Quantum Approximate Optimization Algorithm) encodes each node as a qubit and
builds a parameterized circuit with two alternating layers: a cost layer with parameter
`gamma`, and a mixer layer with parameter `beta`. Measuring gives bitstrings, each a
candidate partition. A classical optimizer then adjusts `gamma` and `beta` to push the
average cut up.

We are splitting an order of pizzas across two ovens. Each edge is a pair of orders that
should not share an oven.


## 1. Check the simulator

```bash
cd /home/ubuntu/tutorial/module2
python3 -c "import qiskit, qiskit_aer; print(qiskit.__version__, qiskit_aer.__version__)"
```

If those are missing:

```bash
pip install --user qiskit qiskit-aer scipy
```


## 2. Run it under Flux

<div class="alert alert-block" style="background-color:#91C2D8;color:#06293D">
<span style="font-weight:600">Description:</span> <code>flux run</code> blocks and streams output, exactly as in Module 1.
</div>

```bash
flux run --cores-per-task=1 python3 scripts/qaoa_maxcut.py -p 2 -s 2048
```

```console
{
  "layers": 2,
  "shots": 2048,
  "evaluations": 40,
  "mean_cut": 4.841,
  "best_bitstring": "01001",
  "best_cut": 6,
  "max_possible_edges": 7,
  "oven_a": [1, 2, 4],
  "oven_b": [0, 3]
}

Oven A: orders [1, 2, 4]
Oven B: orders [0, 3]
Separated 6 of 7 conflicting pairs.
```

Six is the true optimum for this graph, which you can confirm by brute force over all 32
partitions. QAOA found it without enumerating them.


## 3. Sweep the depth

The interesting question is how the answer changes with circuit depth `p`. That is a
parameter sweep, which is the thing Flux is best at:

```bash
flux bulksubmit --watch python3 scripts/qaoa_maxcut.py -p {} ::: 1 2 3 4 5
```

Or use `--cc` to run the same depth repeatedly with different seeds:

```bash
flux submit --cc=1-8 --watch python3 scripts/qaoa_maxcut.py -p 3 --seed {cc}
```

<div class="alert alert-block" style="background-color:#DCECF4;color:#06293D">
<span style="font-weight:600">Notice:</span> More layers is not automatically better. Deeper circuits mean more parameters for the classical optimizer to fit from the same number of shots. This is exactly the kind of trade-off you want a scheduler to let you explore cheaply.
</div>

<div style="background:#DCECF4;border-left:6px solid #D9A441;padding:12px 18px;color:#06293D"><strong>Module 2 complete</strong></div>

## Cleaning up

Module 2 is done, so bring the Usernetes cluster back down:

```bash
kubectl delete all --all
make -C /home/ubuntu/usernetes down
```

Continue to [Module 3](../module3/01_wrapup_and_resources.ipynb).
